# `uniprot` — profile, choose columns & trim

Trim type: **column**. Choose **13 columns**, **5,000 rows**. Save `uniprot_13c_5000r.csv`.

> Output columns: `c0`…`c12`.

In [11]:
import os
import numpy as np
import pandas as pd

NAME       = "uniprot"
N_ROWS     = 10000
N_COLS     = 20
TRIM_ROWS  = "sample"   # head | sample

RAW_PATH   = "uniprot_r539166_c223.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ","
OUT_DELIM  = DELIM
HAS_HEADER = True
ENCODING   = "utf-8"
ON_BAD_LINES = None

## 1. View the raw data

In [12]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (539165, 223)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24', 'c25', 'c26', 'c27', 'c28', 'c29', 'c30', 'c31', 'c32', 'c33', 'c34', 'c35', 'c36', 'c37', 'c38', 'c39', 'c40', 'c41', 'c42', 'c43', 'c44', 'c45', 'c46', 'c47', 'c48', 'c49', 'c50', 'c51', 'c52', 'c53', 'c54', 'c55', 'c56', 'c57', 'c58', 'c59', 'c60', 'c61', 'c62', 'c63', 'c64', 'c65', 'c66', 'c67', 'c68', 'c69', 'c70', 'c71', 'c72', 'c73', 'c74', 'c75', 'c76', 'c77', 'c78', 'c79', 'c80', 'c81', 'c82', 'c83', 'c84', 'c85', 'c86', 'c87', 'c88', 'c89', 'c90', 'c91', 'c92', 'c93', 'c94', 'c95', 'c96', 'c97', 'c98', 'c99', 'c100', 'c101', 'c102', 'c103', 'c104', 'c105', 'c106', 'c107', 'c108', 'c109', 'c110', 'c111', 'c112', 'c113', 'c114', 'c115', 'c116', 'c117', 'c118', 'c119', 'c120', 'c121', 'c122', 'c123', 'c124', 'c125', 'c126', 'c127', 'c128', 'c129', 'c130', 'c131', 'c132', 'c133',

,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9,...,c213,c214,c215,c216,c217,c218,c219,c220,c221,c222
0,Q6GZV8,017L_FRG3G,15165820,Complete proteome; Reference proteome,NaN,predicted,8295; 8316; 8404; 30343; 45438,METMSDYSKEVSEALSALRGELSALSAAISNTVRAGSYSAPVAKDC...,502,53469,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Q6GZT6,040R_FRG3G,15165820,Complete proteome; Reference proteome; Signal,NaN,predicted,8295; 8316; 8404; 30343; 45438,MIRALCTIVLIAAGVAVALYLSLVYGYYMSVGVQDASWLTALTGNR...,182,19577,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Q197A3,057L_IIV3,16912294,Complete proteome; Reference proteome,NaN,predicted,7163; 7183; 42431; 310513; 329105; 332058,MFKIYRTSCMGQHQSQFLHSGTVVQTVDGVTTTSFFQPCLVFPFSI...,130,14846,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q6GZP7,078L_FRG3G,15165820,Complete proteome; Reference proteome,NaN,predicted,8295; 8316; 8404; 30343; 45438,MSIGETFAISAHPEGGALFGTISPGMWNQDFIPWIRKTAVDGHLLI...,212,23985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P0C9E9,1001R_ASFWA,NaN,Complete proteome,NaN,inferred from homology,6937; 9823; 41426; 85517; 273792,MVRLFRNPIKCIFYRRSRKIQEKKLRKSLKKLNFYHPPEDCCQIYR...,124,15327,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
raw.dtypes

c0       object
c1       object
c2       object
c3       object
c4      float64
         ...   
c218     object
c219     object
c220     object
c221     object
c222     object
Length: 223, dtype: object

## 2. Profile: cardinality, top-value %, and group skew

In [14]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

,distinct_values,null_count,null_percent,unique_percent,top_value_percent,mean_group,max_group,skew_ratio
column,,,,,,,,
c0,539165,0,0.00,100.00,0.00,1.00,1,1.00
c1,539165,0,0.00,100.00,0.00,1.00,1,1.00
c2,105531,127739,23.69,19.57,23.69,5.11,127739,25002.65
c3,76455,1232,0.23,14.18,3.30,7.05,17796,2523.55
c4,0,539165,100.00,0.00,100.00,539165.00,539165,1.00
...,...,...,...,...,...,...,...,...
c218,42,539121,99.99,0.01,99.99,12538.72,539121,43.00
c219,28,539137,99.99,0.01,99.99,18591.90,539137,29.00
c220,38,539127,99.99,0.01,99.99,13824.74,539127,39.00


## 3. Choose columns (cardinality mix + id/super-key)

In [15]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

selected columns: ['c0', 'c23', 'c37', 'c41', 'c61', 'c70', 'c71', 'c79', 'c80', 'c81', 'c101', 'c107', 'c111', 'c136', 'c137', 'c142', 'c148', 'c169', 'c176', 'c177']


## 4. Trim to the chosen columns x exact rows

In [16]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

trimmed shape: (10000, 20)


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15,c16,c17,c18,c19
0,C4ZVM1,BWG_2102,C4ZVM1; 20-274,C4ZVM1,NaN,CP001396_GR; BWG_2102,NaN,18273564; VBIEscCol60876_2306,ECOL595496:GI18-2287-MONOMER,NaN,NaN,type: disulfide bond; status: by similarity; ;...,NaN,type: metal ion-binding site; status: by simil...,NaN,NaN,NaN,NaN,NaN,NaN
1,Q5PPJ2,NaN,NaN,Q5PPJ2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Q4UYX6,XC_0668,Q4UYX6; 7-455,Q4UYX6,SM01086; ClpB_D2-small; 1,CP000050_GR; XC_0668,NaN,24062622; VBIXanCam24967_0721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Q8KCW4,CT1293,Q8KCW4; 2-491,NaN,SM00116; CBS; 2,AE006470_GR; CT1293,NaN,21400541; VBIChlTep116050_1178,CTEP194439:GHN0-1287-MONOMER,NaN,NaN,NaN,NaN,type: metal ion-binding site; status: by simil...,NaN,NaN,NaN,NaN,NaN,NaN
4,P34139,NaN,P34139; 5-173,NaN,SM00175; RAB; 1,CM000153_GR; rab1A,NaN,NaN,NaN,type: lipid moiety-binding region; status: by ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Save the trimmed CSV

In [17]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

wrote uniprot_20c_10000r.csv (10000, 20)
reloaded: (10000, 20)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19']


## 6. Check selected cardinality and skew

In [ ]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

In [ ]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof